In [1]:
from __future__ import annotations

%pip install jupyter_events

The `EventLogger` is the main object in Jupyter Events.

In [2]:
from jupyter_events.logger import EventLogger

logger = EventLogger()

To begin emitting events from a Python application, you need to tell the `EventLogger` what events you'd like to emit. To do this, we should register our event's schema (more on this later) with the logger.

In [3]:
schema = """
$id: http://myapplication.org/example-event
version: "1"
title: Example Event
description: An interesting event to collect
properties:
   name:
      title: Name of Event
      type: string
"""


logger.register_event_schema(schema)
print(logger.schemas)

{
  "$id": "http://myapplication.org/example-event",
  "version": "1",
  "title": "Example Event",
  "description": "An interesting event to collect",
  "properties": {
    "name": {
      "title": "Name of Event",
      "type": "string"
    }
  }
}



Now that the logger knows about the event, it needs to know _where_ to send it. To do this, we register a logging _Handler_ —borrowed from Python's standard [`logging`](https://docs.python.org/3/library/logging.html) library—to route the events to the proper place.

In [4]:
# We will import one of the handlers from Python's logging library
from logging import StreamHandler

handler = StreamHandler()

logger.register_handler(handler)

The logger knows about the event and where to send it; all that's left is to emit an instance of the event!

In [5]:
logger.emit(schema_id="http://myapplication.org/example-event", data={"name": "My Event"})

{"__timestamp__": "2025-09-05T08:32:54.425000+00:00Z", "__schema__": "http://myapplication.org/example-event", "__schema_version__": "1", "__metadata_version__": 1, "name": "My Event"}


{'__timestamp__': '2025-09-05T08:32:54.425000+00:00Z',
 '__schema__': 'http://myapplication.org/example-event',
 '__schema_version__': '1',
 '__metadata_version__': 1,
 'name': 'My Event'}

Now, let's demo adding a listener to the already registered event.

In [6]:
async def my_listener(logger: EventLogger, schema_id: str, data: dict) -> None:
    print("hello, from my_listener")
    print(logger)
    print(schema_id)
    print(data)


logger.add_listener(schema_id="http://myapplication.org/example-event", listener=my_listener)

If we emit the event again, you'll see our listener "sees" the event and executes some code:

In [7]:
logger.emit(schema_id="http://myapplication.org/example-event", data={"name": "My Event"})

{"__timestamp__": "2025-09-05T08:33:02.006000+00:00Z", "__schema__": "http://myapplication.org/example-event", "__schema_version__": "1", "__metadata_version__": 1, "name": "My Event"}


{'__timestamp__': '2025-09-05T08:33:02.006000+00:00Z',
 '__schema__': 'http://myapplication.org/example-event',
 '__schema_version__': '1',
 '__metadata_version__': 1,
 'name': 'My Event'}

hello, from my_listener
http://myapplication.org/example-event
{'name': 'My Event'}
